In [81]:
import os
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Metadata

## dirs and files structure

In [82]:
proj_root_path_from_jupyter_run_folder = '..'    # check if this is correct when running the jupyter notebook
proj_root = proj_root_path_from_jupyter_run_folder

# EDA

## ids_relationship.csv

In [83]:
ids_rel_df = pd.read_csv(os.path.join(proj_root, 'data', 'ids_relationship.csv'))

ids_rel_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 275124 entries, 0 to 275123
Data columns (total 3 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   id_ip                  275124 non-null  int64
 1   id_institution         275124 non-null  int64
 2   id_institution_subnet  275124 non-null  int64
dtypes: int64(3)
memory usage: 6.3 MB


In [84]:
ids_rel_df['id_full_subnet'] = ids_rel_df.id_institution.map(str) + '_' + ids_rel_df.id_institution_subnet.map(str)

ids_rel_df.nunique(axis=0)


id_ip                    275124
id_institution              283
id_institution_subnet       548
id_full_subnet              552
dtype: int64

In [85]:
ids_rel_df[['id_institution', 'id_institution_subnet']].value_counts(dropna=False).sort_index().head(20)

id_institution  id_institution_subnet
0               0                         2037
                1                            2
                2                            2
                3                            3
                4                          197
                5                            6
                6                         5111
                7                          124
                8                            6
                9                            5
                10                          27
                11                           1
                12                           1
                13                          35
                14                          42
                15                           8
1               16                          69
                17                       29288
2               18                         252
                19                           4
Name: count, dtype: in

### looks like `id_institution_subnet` is unique, but there are 4 unique pairs, where subnets are shared by several institutes:

In [86]:
ids_rel_df.groupby('id_institution_subnet')['id_institution'].nunique().value_counts(dropna=False)

id_institution
1    544
2      4
Name: count, dtype: int64

#### actually 4 subnets do have second institution - let's see them:

In [87]:
ids_rel_df['nof_inst_per_subnet'] = ids_rel_df.groupby('id_institution_subnet')['id_institution'].transform('nunique')

ids_rel_df.query('nof_inst_per_subnet > 1'
                 )[['id_institution_subnet', 'id_institution']].value_counts(dropna=False).sort_index()

id_institution_subnet  id_institution
264                    69                124
                       279                 1
281                    81                  9
                       121                 8
345                    123                24
                       143                 8
359                    134                 2
                       280                 2
Name: count, dtype: int64

### the true reasons behind are unclear - could be routing plans error, maliсious actions or just dedicated cross-institutional subnets for common distributed tasks.  For the moment we simply save them in a dictionary: 

In [ ]:
tmp_df = ids_rel_df.query('nof_inst_per_subnet > 1').groupby('id_institution_subnet',
                                                             as_index=False)['id_institution'].agg('unique')
tmp_df.id_institution = tmp_df.id_institution.map(lambda npar: [int(n) for n in npar])  # getting rid of numpy types

shared_subnets_dict = dict(zip(tmp_df.id_institution_subnet, tmp_df.id_institution))
joblib.dump(shared_subnets_dict, os.path.join(proj_root, 'py_obj_saves', 'shared_subnets_dict.joblib'))
joblib.load(os.path.join(proj_root, 'py_obj_saves', 'shared_subnets_dict.joblib'))

{264: [69, 279], 281: [81, 121], 345: [123, 143], 359: [134, 280]}